# Revela — Notebook 06: BCN20000 Cancer-Risk CNN Training

**Issue:** #119 — D3.4: Retrain dermoscopic CNN with cancer-risk taxonomy  
**Status:** In progress  
**Depends on:** #118 — BCN20000 splits for cancer-risk taxonomy (closed)  
**Script:** `src/model/train.py`  
**Config:** `config/bcn20000_cancer_risk_config.yaml`  

### Purpose
Train an EfficientNet-B0 baseline on the 4-class cancer-risk taxonomy using the BCN20000 splits built in #118. This notebook covers:
- Training configuration and class weights
- Per-class recall logic (melanoma, NMSC, combined cancer recall)
- Smoke test verification
- Full training run results
- Training history and priority metrics

### Deliverables
- `models/bcn20000_cancer_risk_effnet_b0/best_model.pth`
- `models/bcn20000_cancer_risk_effnet_b0/class_to_idx.json`
- `models/bcn20000_cancer_risk_effnet_b0/training_history.csv`
- `docs/model/bcn20000_cancer_risk_training_summary.md`

### Priority metrics
1. Validation macro-F1
2. Cancer / malignant recall (Melanoma + NMSC combined)
3. Melanoma recall
4. Non-melanoma skin cancer recall
5. Balanced accuracy

> Old CNN v1 artifacts in `models/effnet_b0/` are untouched.

## Block 1 — Training Pipeline Overview

| Item | Detail |
|------|--------|
| Entry point | `python3 -m src.model.train` |
| Config | `config/bcn20000_cancer_risk_config.yaml` |
| Train CSV | `data/processed/bcn20000_cancer_risk/train.csv` (12,352 rows) |
| Val CSV | `data/processed/bcn20000_cancer_risk/val.csv` (2,628 rows) |
| Architecture | EfficientNet-B0, ImageNet pretrained |
| Classes | 4 (Melanoma / Non-melanoma skin cancer / Benign nevus / Other non-cancer) |
| Epochs | 10 |
| Batch size | 16 |
| Learning rate | 0.0003 |
| Weight decay | 0.01 |
| Class weighting | Inverse frequency (`total / (num_classes × count)`) |
| Best model criterion | Validation macro-F1 |
| Output directory | `models/bcn20000_cancer_risk_effnet_b0/` |

The existing `src/model/train.py` supports this config without changes — it reads `class_names` and `training.output_dir` from the YAML.

## Block 2 — Config Review

In [ ]:
import yaml

with open('../../../config/bcn20000_cancer_risk_config.yaml') as f:
    config = yaml.safe_load(f)

print(yaml.dump(config, default_flow_style=False, sort_keys=False).strip())

dataset:
  metadata_csv: revela/bcn20000_metadata_2026-05-11.csv
  image_root: revela/BCN20000
  output_dir: data/processed/bcn20000_cancer_risk
  summary_path: docs/model/bcn20000_cancer_risk_split_summary.md
  image_extension: .jpg

columns:
  image_id: isic_id
  lesion_id: lesion_id
  diagnosis: diagnosis_3

class_names:
  - Melanoma
  - Non-melanoma skin cancer
  - Benign nevus
  - Other non-cancer / indeterminate lesion

split:
  train: 0.70
  val: 0.15
  test: 0.15
  seed: 42

training:
  image_size: 224
  batch_size: 16
  num_workers: 0
  learning_rate: 0.0003
  weight_decay: 0.01
  epochs: 10
  use_class_weights: true
  output_dir: models/bcn20000_cancer_risk_effnet_b0

files:
  master_metadata: master_metadata.csv
  train: train.csv
  val: val.csv
  test: test.csv


## Block 3 — Class Weights Preview

Class weights are computed as `total_train / (num_classes × class_count)`.  
Higher weight = underrepresented class = larger gradient penalty on misclassification.

In [ ]:
import pandas as pd

BASE = '../../../data/processed/bcn20000_cancer_risk'
CLASS_ORDER = [
    'Melanoma',
    'Non-melanoma skin cancer',
    'Benign nevus',
    'Other non-cancer / indeterminate lesion',
]

train = pd.read_csv(f'{BASE}/train.csv')
total = len(train)
num_classes = len(CLASS_ORDER)

print('=== TRAINING SPLIT CLASS DISTRIBUTION & WEIGHTS ===')
print()
print(f'{"Class":<48} {"Count":>5}  {"   %":>5}  {"Weight":>8}')
print('-' * 70)

weights = {}
for cls in CLASS_ORDER:
    count = (train['class_label'] == cls).sum()
    pct = count / total * 100
    w = total / (num_classes * count)
    weights[cls] = w
    print(f'{cls:<48} {count:>5}  {pct:>5.1f}%  {w:>8.4f}')

print('-' * 70)
print(f'{"TOTAL":<48} {total:>5} {100.0:>5.1f}%')
print()

max_cls = max(weights, key=weights.get)
min_cls = min(weights, key=weights.get)
print(f'Most underrepresented: {max_cls} (weight {weights[max_cls]:.4f})')
print(f'Most overrepresented : {min_cls} (weight {weights[min_cls]:.4f})')
print()

cancer_count = sum((train['class_label'] == c).sum() for c in ['Melanoma', 'Non-melanoma skin cancer'])
noncancer_count = total - cancer_count
print(f'Cancer / malignant (Melanoma + NMSC) : {cancer_count} samples ({cancer_count/total*100:.1f}% of train)')
print(f'Non-cancer (Benign nevus + Other)    : {noncancer_count} samples ({noncancer_count/total*100:.1f}% of train)')

=== TRAINING SPLIT CLASS DISTRIBUTION & WEIGHTS ===

Class                                           Count     %    Weight
----------------------------------------------------------------------
Melanoma                                         3363  27.2%    0.9182
Non-melanoma skin cancer                         2968  24.0%    1.0404
Benign nevus                                     3934  31.8%    0.7850
Other non-cancer / indeterminate lesion          2087  16.9%    1.4796
----------------------------------------------------------------------
TOTAL                                           12352 100.0%

Most underrepresented: Other non-cancer / indeterminate lesion (weight 1.4796)
Most overrepresented : Benign nevus (weight 0.7850)

Cancer / malignant (Melanoma + NMSC) : 6331 samples (51.3% of train)
Non-cancer (Benign nevus + Other)    : 6021 samples (48.7% of train)


## Block 4 — Per-Class Recall Logic

The existing `src/model/train.py` reports macro-F1 and balanced accuracy but not per-class recalls.  
This block defines and verifies the logic that will be added to `train.py`.

Three metrics needed for #119:
- **Melanoma recall** — of all true melanoma cases, how many did the model catch?
- **NMSC recall** — same for non-melanoma skin cancer
- **Cancer recall** — of all malignant cases (melanoma + NMSC combined), how many did the model identify as *any* cancer class?

In [ ]:
def compute_per_class_recalls(
    true_labels: list[int],
    predicted_labels: list[int],
    num_classes: int,
) -> list[float]:
    """Return per-class recall as a list indexed by class index."""
    recalls = []
    for class_index in range(num_classes):
        class_total = sum(1 for t in true_labels if t == class_index)
        class_correct = sum(
            1 for t, p in zip(true_labels, predicted_labels)
            if t == class_index and p == class_index
        )
        recalls.append(class_correct / class_total if class_total > 0 else 0.0)
    return recalls


_CANCER_CLASS_NAMES = {'Melanoma', 'Non-melanoma skin cancer'}

def compute_cancer_recall(
    true_labels: list[int],
    predicted_labels: list[int],
    class_to_idx: dict[str, int],
) -> float | None:
    """
    Fraction of true malignant cases (Melanoma or NMSC) that were predicted
    as any malignant class. Returns None if fewer than 2 cancer classes exist.
    """
    cancer_indices = {idx for cls, idx in class_to_idx.items() if cls in _CANCER_CLASS_NAMES}
    if len(cancer_indices) < 2:
        return None
    total = sum(1 for t in true_labels if t in cancer_indices)
    correct = sum(
        1 for t, p in zip(true_labels, predicted_labels)
        if t in cancer_indices and p in cancer_indices
    )
    return correct / total if total > 0 else 0.0


# --- Sanity checks ---
CLASS_TO_IDX = {
    'Melanoma': 0,
    'Non-melanoma skin cancer': 1,
    'Benign nevus': 2,
    'Other non-cancer / indeterminate lesion': 3,
}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}
N = 4

true = [0, 0, 1, 1, 2, 2, 3, 3]

print('=== PER-CLASS RECALL FUNCTION ===')
print()
print('compute_per_class_recalls: list of recall per class, indexed by class_to_idx')
print('compute_cancer_recall    : fraction of true cancer cases predicted as any cancer class')
print()

print('=== SANITY CHECK (perfect predictions) ===')
perfect_preds = true[:]
recalls = compute_per_class_recalls(true, perfect_preds, N)
for idx, r in enumerate(recalls):
    print(f'  {IDX_TO_CLASS[idx]} recall{" ":>{40 - len(IDX_TO_CLASS[idx])}}: {r:.4f}  PASS')
cr = compute_cancer_recall(true, perfect_preds, CLASS_TO_IDX)
print(f'  Cancer recall                          : {cr:.4f}  PASS')
print()

print('=== SANITY CHECK (all predicted as Melanoma) ===')
all_mel_preds = [0] * len(true)
recalls = compute_per_class_recalls(true, all_mel_preds, N)
for idx, r in enumerate(recalls):
    note = '(all melanoma caught)' if idx == 0 else ('(all NMSC missed)' if idx == 1 else '')
    print(f'  {IDX_TO_CLASS[idx]} recall{" ":>{40 - len(IDX_TO_CLASS[idx])}}: {r:.4f}  {note}')
cr = compute_cancer_recall(true, all_mel_preds, CLASS_TO_IDX)
print(f'  Cancer recall                          : {cr:.4f}  (NMSC pred as Melanoma still counts as cancer)')

=== PER-CLASS RECALL FUNCTION ===

compute_per_class_recalls: list of recall per class, indexed by class_to_idx
compute_cancer_recall    : fraction of true cancer cases predicted as any cancer class

=== SANITY CHECK (perfect predictions) ===
  Melanoma recall                        : 1.0000  PASS
  Non-melanoma skin cancer recall        : 1.0000  PASS
  Benign nevus recall                    : 1.0000  PASS
  Other non-cancer / indeterminate lesion recall : 1.0000  PASS
  Cancer recall                          : 1.0000  PASS

=== SANITY CHECK (all predicted as Melanoma) ===
  Melanoma recall                        : 1.0000  (all melanoma caught)
  Non-melanoma skin cancer recall        : 0.0000  (all NMSC missed)
  Benign nevus recall                    : 0.0000
  Other non-cancer / indeterminate lesion recall : 0.0000
  Cancer recall                          : 1.0000  (NMSC pred as Melanoma still counts as cancer)


## Block 5 — Smoke Test

Run 1 epoch with 2 train batches and 2 val batches.  
Purpose: verify config parsing, image loading, EfficientNet-B0 init, metric writing, checkpoint creation.  
These metrics are meaningless — they only confirm the pipeline runs end-to-end.

```bash
python3 -m src.model.train \
  --config config/bcn20000_cancer_risk_config.yaml \
  --epochs 1 \
  --batch-size 4 \
  --num-workers 0 \
  --max-train-batches 2 \
  --max-val-batches 2
```

In [ ]:
import subprocess

result = subprocess.run(
    [
        'python3', '-m', 'src.model.train',
        '--config', 'config/bcn20000_cancer_risk_config.yaml',
        '--epochs', '1',
        '--batch-size', '4',
        '--num-workers', '0',
        '--max-train-batches', '2',
        '--max-val-batches', '2',
    ],
    capture_output=True,
    text=True,
    cwd='../../..',  # run from repo root
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

## Block 6 — Full Training Run

10 epochs, all train/val batches, MPS (Apple Silicon) or CPU.

```bash
python3 -m src.model.train --config config/bcn20000_cancer_risk_config.yaml
```

Expected output directory: `models/bcn20000_cancer_risk_effnet_b0/`

In [ ]:
import subprocess

result = subprocess.run(
    [
        'python3', '-m', 'src.model.train',
        '--config', 'config/bcn20000_cancer_risk_config.yaml',
    ],
    capture_output=True,
    text=True,
    cwd='../../..',
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

## Block 7 — Training History

Load `training_history.csv` written by the training run and display the per-epoch metrics table.

In [ ]:
import pandas as pd

history = pd.read_csv('../../../models/bcn20000_cancer_risk_effnet_b0/training_history.csv')

print('=== TRAINING HISTORY ===')
print(history.to_string(index=False, float_format='{:.4f}'.format))
print()
print(f'Best val macro-F1 : {history["val_macro_f1"].max():.4f}  (epoch {history["val_macro_f1"].idxmax() + 1})')
print(f'Best val balanced accuracy : {history["val_balanced_accuracy"].max():.4f}')

## Block 8 — Per-Class Recall From Training History

Extract melanoma recall, NMSC recall, and combined cancer recall from the saved history.  
These columns are present only if `train.py` was updated to log them (see Block 4 for the logic).

In [ ]:
import pandas as pd

history = pd.read_csv('../../../models/bcn20000_cancer_risk_effnet_b0/training_history.csv')

RECALL_COLS = [
    'val_recall_melanoma',
    'val_recall_non_melanoma_skin_cancer',
    'val_cancer_recall',
    'val_balanced_accuracy',
    'val_macro_f1',
]

available = [c for c in RECALL_COLS if c in history.columns]
missing   = [c for c in RECALL_COLS if c not in history.columns]

if missing:
    print(f'NOTE: {missing} not in history — update train.py with Block 4 logic first.')

if available:
    print('=== PRIORITY METRICS BY EPOCH ===')
    print(history[['epoch'] + available].to_string(index=False, float_format='{:.4f}'.format))
    print()
    best_epoch = history['val_macro_f1'].idxmax()
    print('Best epoch results:')
    for col in available:
        print(f'  {col:<40}: {history.loc[best_epoch, col]:.4f}')

## Block 9 — Required train.py Changes

The Block 4 logic needs to be wired into `src/model/train.py` so the metrics are saved during training.  
Changes required:

1. **Add** `compute_per_class_recalls()` and `compute_cancer_recall()` functions (code from Block 4)
2. **Refactor** `compute_balanced_accuracy()` to call `compute_per_class_recalls()` instead of duplicating the loop
3. **Update** `evaluate()` signature to `(model, loader, criterion, device, num_classes, class_to_idx=None, max_batches=None)` — return `(loss, acc, macro_f1, balanced_acc, per_class_recalls, cancer_recall)`
4. **Update** `save_training_history()` to derive fieldnames from `history_rows[0].keys()` instead of the hardcoded list
5. **Update** main training loop to:
   - Unpack new return values from `evaluate()`
   - Build per-class recall column names with `val_recall_<normalized_class_name>`
   - Add `val_cancer_recall` when both cancer classes are present
   - Print per-class recalls each epoch

Column naming convention (to match Block 8 above):

```python
import re
def _recall_col(class_name: str) -> str:
    return 'val_recall_' + re.sub(r'[^a-z0-9]+', '_', class_name.lower()).strip('_')
```

Expected new columns in `training_history.csv`:
- `val_recall_melanoma`
- `val_recall_non_melanoma_skin_cancer`
- `val_recall_benign_nevus`
- `val_recall_other_non_cancer_indeterminate_lesion`
- `val_cancer_recall`

## Block 10 — Summary

| Item | Detail |
|------|--------|
| Issue | #119 — D3.4 |
| Taxonomy | 4-class cancer-risk |
| Architecture | EfficientNet-B0, ImageNet pretrained |
| Imbalance strategy | Inverse frequency class weights |
| Best model criterion | Validation macro-F1 |
| Output directory | `models/bcn20000_cancer_risk_effnet_b0/` |

### Deliverable checklist

- [ ] Smoke test passes (Block 5)
- [ ] `train.py` updated with per-class recall tracking (Block 9)
- [ ] Full training run complete (Block 6)
- [ ] `models/bcn20000_cancer_risk_effnet_b0/best_model.pth` saved
- [ ] `models/bcn20000_cancer_risk_effnet_b0/class_to_idx.json` saved
- [ ] `models/bcn20000_cancer_risk_effnet_b0/training_history.csv` saved
- [ ] Training history loaded and priority metrics reviewed (Block 7, 8)
- [ ] `docs/model/bcn20000_cancer_risk_training_summary.md` written

### Dependency chain
```
D3.1 — Approve cancer-risk taxonomy    ✅ (#116)
  → D3.2 (#117) — Create label mapping  ✅
    → D3.3 (#118) — Rebuild splits       ✅
      → D3.4 (#119) — Retrain CNN         ← THIS NOTEBOOK
        → D3.5 (#120) — Evaluate CNN
```

### Next step
#120 — Evaluate the trained model on the held-out test split with class-wise metrics and a confusion matrix.